# digital-twin-for-ipa — test in Google Colab

> A machine-readable **digital twin of India's Investment Promotion Apparatus** — the full set of central-government entities, the incentive instruments each one actually offers companies, how the instruments interlink (stack / exclude / converge / feed), what the budget record says about delivery, an

## What this notebook does
Clones the repo, installs its dependencies, and runs a **code smoke test** (byte-compiles every module and imports the entry points) so you can confirm the code is sound before running it for real.

## Requirements

**Python packages:** `pypdf>=4.0          # text extraction from scanned-layout PDFs the WebFetch converter rejects (e.g. MIB guidelines), pandas>=2.0         # ad-hoc analysis over layers/13_flat_instrument_index.json and state/ snapshots`

**Entry points detected:**
- `scripts/build_clearance_leads.py`
- `scripts/build_target_leads.py`
- `scripts/merge_cluster_leads.py`
- `scripts/pib_visibility.py`
- `scripts/refresh_twin.py`
- `scripts/send_twin_alert.py`

## Before you run for real — caveats
- **Needs credentials/secrets** (API keys, .env, or mailer/broker logins) — set these in the Colab session before running the relevant scripts; they are gitignored and not in the repo.
- **Needs data files** (DuckDB/parquet/cache). Some are committed (incl. via Git LFS — run `!git lfs pull` in Colab); others are generated by the collectors or fetched live.

*Generated testing scaffold — edit freely.*


In [ ]:
# Clone the repo
!git clone --depth 1 https://github.com/herrrickshaw/digital-twin-for-ipa.git
%cd digital-twin-for-ipa
!git lfs pull || true

In [ ]:
# Install dependencies
import os
if os.path.exists('requirements.txt'):
    !pip install -q -r requirements.txt
elif os.path.exists('pyproject.toml'):
    !pip install -q -e .
else:
    print('No requirements.txt / pyproject.toml — nothing to install')

In [ ]:
# Code smoke test: byte-compile every module (syntax/def-level check)
import subprocess, sys, glob
pyfiles = glob.glob('**/*.py', recursive=True)
print(f'Compiling {len(pyfiles)} Python files...')
r = subprocess.run([sys.executable, '-m', 'py_compile', *pyfiles],
                   text=True, capture_output=True)
print('OK — all modules compile' if r.returncode == 0
      else 'Compile errors:\n' + r.stderr)

In [ ]:
# List entry points and their docstrings — how to run each
import ast, glob
for f in sorted(glob.glob('*.py') + glob.glob('collectors/*.py')
                + glob.glob('scripts/*.py') + glob.glob('src/*.py')):
    try:
        doc = ast.get_docstring(ast.parse(open(f).read())) or ''
    except Exception:
        doc = ''
    print(f'\n=== {f} ===')
    print(doc.strip()[:400])
    print(f'Run with:  !python {f} --help')